# Q3: Sandpile Model (BTW Model)

**Roll Number:** 203101108  
**Course:** UAS Spring'26

---

## Problem Statement

Implement the Abelian Sandpile Model (Bak-Tang-Wiesenfeld model) on a 2D lattice:

1. **Initial Setup:** Set all lattice sites to the avalanche threshold
2. **Relaxation:** Run until the system stabilizes (all sites below threshold)
3. **Plot** the final stable pattern
4. **Perturbation:** Add a grain at the center and track evolution at $t = 0, 10, 20$, and final state
5. **Explain** the observed dynamics

## Parameter Calculations

For **Roll Number = 203101108**:

$$\text{Roll Number} \mod 7 = 203101108 \mod 7 = 0$$
$$\text{Roll Number} \mod 5 = 203101108 \mod 5 = 3$$

| Parameter | Formula | Value |
|-----------|---------|-------|
| Lattice Size $L$ | $((\text{roll} \mod 7) + 1) \times 10$ | **10** |
| Grid Dimensions | $L \times L$ | **10 × 10** |
| Avalanche Threshold | $(\text{roll} \mod 5) + 3$ | **6** |

---

## Sandpile Model Theory

### The BTW Model

The **Bak-Tang-Wiesenfeld (BTW) sandpile model** is a paradigmatic example of **self-organized criticality (SOC)**.

### Rules

1. Each lattice site $(i, j)$ has a height $z(i,j)$ representing the number of "grains"

2. **Toppling Rule:** If $z(i,j) \geq z_c$ (threshold), the site topples:
   - $z(i,j) \rightarrow z(i,j) - 4$
   - Each of the 4 neighbors gains 1 grain

3. **Open Boundary Condition:** Grains that would go outside the grid are **lost** (fall off the edge)
   - Corner sites: lose 2 grains when toppling
   - Edge sites: lose 1 grain when toppling
   - Interior sites: conserve all 4 grains

4. **Parallel Update:** In each time step, ALL unstable sites topple simultaneously

5. **Relaxation:** Continue until all sites have $z < z_c$

---

## Setup and Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import matplotlib.patches as mpatches

# For better plot quality
plt.rcParams['figure.dpi'] = 150
plt.rcParams['savefig.dpi'] = 150
plt.rcParams['font.size'] = 11

# Roll number parameters
ROLL_NUMBER = 203101108

L = ((ROLL_NUMBER % 7) + 1) * 10  # Lattice size
THRESHOLD = (ROLL_NUMBER % 5) + 3  # Avalanche threshold

print(f"Roll Number: {ROLL_NUMBER}")
print(f"Roll Number % 7 = {ROLL_NUMBER % 7}")
print(f"Roll Number % 5 = {ROLL_NUMBER % 5}")
print(f"\nParameters:")
print(f"  Lattice Size L = {L}")
print(f"  Grid: {L} × {L}")
print(f"  Avalanche Threshold z_c = {THRESHOLD}")

---

## Core Sandpile Functions

In [ ]:
def topple_step(grid, threshold):
    """
    Perform one parallel toppling sweep.
    All sites with z >= threshold topple simultaneously.
    
    Uses OPEN boundary conditions:
    - Grains going outside the grid are lost
    
    Returns:
        new_grid: Updated grid after toppling
        num_toppled: Number of sites that toppled
    """
    L = grid.shape[0]
    
    # Find unstable sites
    unstable = grid >= threshold
    num_toppled = np.sum(unstable)
    
    if num_toppled == 0:
        return grid.copy(), 0
    
    # Create new grid
    new_grid = grid.copy()
    
    # Each unstable site loses 4 grains
    new_grid[unstable] -= 4
    
    # Distribute grains to neighbors (open boundary - grains at edge are lost)
    # Top neighbor (i-1, j)
    new_grid[1:, :] += unstable[:-1, :].astype(int)
    # Bottom neighbor (i+1, j)
    new_grid[:-1, :] += unstable[1:, :].astype(int)
    # Left neighbor (i, j-1)
    new_grid[:, 1:] += unstable[:, :-1].astype(int)
    # Right neighbor (i, j+1)
    new_grid[:, :-1] += unstable[:, 1:].astype(int)
    
    return new_grid, num_toppled


def relax(grid, threshold, max_steps=10000):
    """
    Run the sandpile until it reaches a stable state.
    
    Returns:
        final_grid: Stable configuration
        total_steps: Number of toppling sweeps performed
        history: List of (step, num_toppled) tuples
    """
    current_grid = grid.copy()
    history = []
    
    for step in range(max_steps):
        current_grid, num_toppled = topple_step(current_grid, threshold)
        history.append((step, num_toppled))
        
        if num_toppled == 0:
            break
    
    return current_grid, step, history


def plot_sandpile(grid, threshold, title, filename=None, show_values=True):
    """
    Plot the sandpile configuration with a nice colormap.
    """
    fig, ax = plt.subplots(figsize=(8, 7))
    
    # Create colormap for values 0 to threshold
    colors = ['#f7fbff', '#deebf7', '#c6dbef', '#9ecae1', '#6baed6', '#3182bd', '#08519c']
    n_colors = threshold + 2  # 0 to threshold+1 (for overflow visualization)
    cmap = plt.cm.Blues
    
    # Plot
    im = ax.imshow(grid, cmap='YlOrRd', vmin=0, vmax=threshold, interpolation='nearest')
    
    # Add colorbar
    cbar = plt.colorbar(im, ax=ax, label='Height z(i,j)')
    cbar.set_ticks(range(threshold + 1))
    
    # Add grid lines
    ax.set_xticks(np.arange(-0.5, grid.shape[1], 1), minor=True)
    ax.set_yticks(np.arange(-0.5, grid.shape[0], 1), minor=True)
    ax.grid(which='minor', color='gray', linestyle='-', linewidth=0.5, alpha=0.5)
    
    # Show values in cells if grid is small enough
    if show_values and grid.shape[0] <= 15:
        for i in range(grid.shape[0]):
            for j in range(grid.shape[1]):
                val = grid[i, j]
                color = 'white' if val > threshold/2 else 'black'
                ax.text(j, i, str(val), ha='center', va='center', 
                       fontsize=8, color=color, fontweight='bold')
    
    ax.set_xlabel('Column j', fontsize=11)
    ax.set_ylabel('Row i', fontsize=11)
    ax.set_title(title, fontsize=12)
    
    # Set ticks
    ax.set_xticks(range(grid.shape[1]))
    ax.set_yticks(range(grid.shape[0]))
    
    plt.tight_layout()
    
    if filename:
        plt.savefig(filename, dpi=150, bbox_inches='tight')
        print(f"Figure saved: {filename}")
    
    plt.show()
    
    return fig

---

## Part 1: Initial Relaxation

We start by setting **all sites to the threshold value** ($z_c = 6$) and let the system relax to a stable state.

In [ ]:
# Initialize all sites at threshold
initial_grid = np.full((L, L), THRESHOLD, dtype=int)

print(f"Initial Configuration:")
print(f"  All sites set to z = {THRESHOLD} (threshold)")
print(f"  Grid size: {L} × {L}")
print(f"  Total grains: {np.sum(initial_grid)}")
print(f"\nStarting relaxation...")

# Run relaxation
final_grid, total_steps, history = relax(initial_grid, THRESHOLD)

print(f"\nRelaxation Complete!")
print(f"  Time steps to stabilize: {total_steps}")
print(f"  Total grains remaining: {np.sum(final_grid)}")
print(f"  Grains lost at boundary: {np.sum(initial_grid) - np.sum(final_grid)}")
print(f"  Max height in final state: {np.max(final_grid)}")
print(f"  Min height in final state: {np.min(final_grid)}")

In [ ]:
# Plot the final stable state after initial relaxation
plot_sandpile(final_grid, THRESHOLD, 
              f'Final Stable State After Initial Relaxation\n(Started from all sites at z = {THRESHOLD}, L = {L}×{L})',
              'figures/initial_relaxation.png')

### Interpretation of Initial Relaxation

**Observations:**

1. **Starting state:** All 100 sites at $z = 6$ (threshold), totaling 600 grains

2. **Massive avalanche:** Since every site starts at threshold, the entire system is unstable. A massive cascade of topplings occurs.

3. **Grain loss:** Grains are progressively lost at the open boundaries until the system stabilizes.

4. **Final pattern:** The stable configuration shows a characteristic pattern with:
   - Lower values near the edges (grains have escaped)
   - Higher values toward the center (harder for grains to escape)
   - All values strictly below threshold ($z < 6$)

5. **Self-organized criticality:** The system has naturally evolved to a **critical state** where it's stable but sensitive to small perturbations.

---

## Part 2: Perturbation and Evolution

Now we add a **single grain at the center** of the stable configuration and observe the avalanche dynamics.

In [ ]:
# Start from the relaxed state
perturbed_grid = final_grid.copy()

# Add 1 grain at the center
center = L // 2
print(f"Adding 1 grain at center position ({center}, {center})")
print(f"Height before: {perturbed_grid[center, center]}")

perturbed_grid[center, center] += 1

print(f"Height after: {perturbed_grid[center, center]}")
print(f"Threshold: {THRESHOLD}")
print(f"Site is unstable: {perturbed_grid[center, center] >= THRESHOLD}")

In [ ]:
# Record states at specific time steps
states = {}
current_grid = perturbed_grid.copy()

# t = 0: Just after perturbation
states[0] = current_grid.copy()

# Run simulation and record at t = 10, 20
time_points = [10, 20]
max_time = 100  # Maximum steps to run

print("\nRunning simulation after perturbation...")
print(f"{'Time':>6} | {'Sites Toppled':>14} | {'Max Height':>10}")
print("-" * 40)

total_toppled = 0
for t in range(1, max_time + 1):
    current_grid, num_toppled = topple_step(current_grid, THRESHOLD)
    total_toppled += num_toppled
    
    if t <= 25 or t in time_points or num_toppled == 0:
        print(f"{t:>6} | {num_toppled:>14} | {np.max(current_grid):>10}")
    
    if t in time_points:
        states[t] = current_grid.copy()
    
    if num_toppled == 0:
        print(f"\nSystem stabilized at t = {t}")
        break

# Final state
states['final'] = current_grid.copy()
final_time = t

print(f"\nTotal sites toppled during avalanche: {total_toppled}")
print(f"Time to stabilize: {final_time} steps")

In [ ]:
# Plot all four states in a 2x2 grid
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

plot_configs = [
    (0, 't = 0 (Just After Perturbation)'),
    (10, 't = 10 (During Avalanche)'),
    (20, 't = 20 (Avalanche Propagation)'),
    ('final', f't = {final_time} (Final Stable State)')
]

for ax, (t, title) in zip(axes.flat, plot_configs):
    grid = states[t]
    
    im = ax.imshow(grid, cmap='YlOrRd', vmin=0, vmax=THRESHOLD, interpolation='nearest')
    
    # Add grid lines
    ax.set_xticks(np.arange(-0.5, L, 1), minor=True)
    ax.set_yticks(np.arange(-0.5, L, 1), minor=True)
    ax.grid(which='minor', color='gray', linestyle='-', linewidth=0.5, alpha=0.5)
    
    # Show values
    for i in range(L):
        for j in range(L):
            val = grid[i, j]
            color = 'white' if val > THRESHOLD/2 else 'black'
            ax.text(j, i, str(val), ha='center', va='center', 
                   fontsize=7, color=color, fontweight='bold')
    
    # Mark center
    if t == 0:
        ax.plot(center, center, 'g*', markersize=15, markeredgecolor='black', markeredgewidth=1)
    
    ax.set_xlabel('Column j')
    ax.set_ylabel('Row i')
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_xticks(range(L))
    ax.set_yticks(range(L))

# Add colorbar
fig.subplots_adjust(right=0.9)
cbar_ax = fig.add_axes([0.92, 0.15, 0.02, 0.7])
cbar = fig.colorbar(im, cax=cbar_ax, label='Height z(i,j)')
cbar.set_ticks(range(THRESHOLD + 1))

fig.suptitle(f'Sandpile Evolution After Center Perturbation\n(L = {L}×{L}, Threshold = {THRESHOLD})', 
             fontsize=14, fontweight='bold', y=1.02)

plt.tight_layout()
plt.savefig('figures/perturbation_evolution.png', dpi=150, bbox_inches='tight')
print("Figure saved: figures/perturbation_evolution.png")
plt.show()

In [ ]:
# Also save individual figures for the report
for t, title_suffix in [(0, 't0'), (10, 't10'), (20, 't20'), ('final', 'final')]:
    if t in states:
        grid = states[t]
        
        if t == 'final':
            title = f'Final Stable State (t = {final_time})'
        else:
            title = f'Sandpile State at t = {t}'
        
        plot_sandpile(grid, THRESHOLD, 
                     f'{title}\n(After Center Perturbation, L = {L}×{L})',
                     f'figures/perturbation_{title_suffix}.png')

---

## Interpretation of Perturbation Evolution

### Figure Analysis

**t = 0 (Just After Perturbation):**
- One grain added at center position (5, 5)
- If the center site was at height 5, it becomes 6 (= threshold), triggering an avalanche
- The green star marks the perturbation location

**t = 10 (During Avalanche):**
- The avalanche has propagated outward from the center
- Sites near the center have redistributed their grains
- The wavefront of instability spreads in all directions

**t = 20 (Avalanche Propagation):**
- Further propagation of the disturbance
- Grains reaching the boundary begin to fall off
- The pattern shows the characteristic "ripple" effect of sandpile dynamics

**Final State:**
- System has returned to a stable configuration
- All heights are below threshold (z < 6)
- The pattern may differ from the original stable state due to the avalanche

### Key Physics

1. **Sensitivity to perturbation:** A single grain can trigger a cascade of topplings

2. **Spatial propagation:** The avalanche spreads outward from the perturbation site

3. **Boundary effects:** Grains are lost at the open boundaries, allowing the system to eventually stabilize

4. **Self-organized criticality:** The system naturally returns to a critical state after the perturbation

---

## Additional Analysis: Difference Maps

In [ ]:
# Show the difference between initial stable state and final state after perturbation
diff_grid = states['final'] - final_grid

fig, ax = plt.subplots(figsize=(8, 7))

# Use diverging colormap
max_diff = max(abs(diff_grid.min()), abs(diff_grid.max()))
if max_diff == 0:
    max_diff = 1  # Avoid division by zero

im = ax.imshow(diff_grid, cmap='RdBu_r', vmin=-max_diff, vmax=max_diff, interpolation='nearest')

# Add grid lines
ax.set_xticks(np.arange(-0.5, L, 1), minor=True)
ax.set_yticks(np.arange(-0.5, L, 1), minor=True)
ax.grid(which='minor', color='gray', linestyle='-', linewidth=0.5, alpha=0.5)

# Show values
for i in range(L):
    for j in range(L):
        val = diff_grid[i, j]
        if val != 0:
            color = 'white' if abs(val) > max_diff/2 else 'black'
            ax.text(j, i, f'{val:+d}', ha='center', va='center', 
                   fontsize=8, color=color, fontweight='bold')

plt.colorbar(im, ax=ax, label='Change in Height')
ax.set_xlabel('Column j')
ax.set_ylabel('Row i')
ax.set_title(f'Difference Map: Final State After Perturbation - Initial Stable State\n(Red = gained grains, Blue = lost grains)')
ax.set_xticks(range(L))
ax.set_yticks(range(L))

plt.tight_layout()
plt.savefig('figures/difference_map.png', dpi=150, bbox_inches='tight')
print("Figure saved: figures/difference_map.png")
plt.show()

print(f"\nTotal change in grains: {np.sum(diff_grid)}")
print(f"(Negative means grains were lost to boundary)")

---

## Conclusions

### Summary of Results

| Aspect | Value/Observation |
|--------|-------------------|
| Lattice Size | 10 × 10 |
| Threshold | 6 |
| Initial Relaxation Steps | See output above |
| Perturbation Avalanche Duration | See output above |
| Boundary Condition | Open (grains lost at edges) |

### Key Physical Insights

1. **Self-Organized Criticality (SOC):**
   - The sandpile naturally evolves to a critical state
   - No external tuning required—criticality emerges from the dynamics

2. **Avalanche Dynamics:**
   - A single grain perturbation can trigger cascades of varying sizes
   - Avalanche sizes in the BTW model follow power-law distributions

3. **Open Boundary Importance:**
   - Without grain loss at boundaries, the system would never stabilize
   - The boundary acts as a "sink" for excess grains

4. **Spatial Structure:**
   - The stable state shows characteristic patterns
   - Heights tend to be lower near edges and higher toward the center

### Broader Significance

The BTW sandpile model demonstrates that complex, scale-free behavior can emerge from simple local rules. This paradigm of **self-organized criticality** has applications in:
- Earthquake dynamics
- Forest fires
- Neural avalanches in the brain
- Stock market fluctuations